# JARVIS-DFT Thermoelectric Data Extraction
Downloads the JARVIS `dft_3d` database and splits it into two families:
1. **Bismuth-telluride / tetradymite family** (Bi₂Te₃, Sb₂Te₃, Bi₂Se₃, and related Bi-Sb-Te-Se chalcogenides)
2. **Half-Heusler family** (ABC ternary compounds, 1:1:1 stoichiometry, space group F-43m #216)

Run cells top to bottom. First run will download and cache the dataset (a few hundred MB).

## 1. Install dependencies

In [ ]:
!pip install jarvis-tools pandas -q

## 2. Download the JARVIS dft_3d dataset

In [ ]:
import pandas as pd
from jarvis.db.figshare import data as jdata

print("Downloading JARVIS dft_3d dataset (first run only, then cached)...")
d = jdata("dft_3d")   # swap for "dft_3d_2021" / "dft_3d_2022" for a fixed snapshot
df = pd.DataFrame(d)
print(f"Total entries in dft_3d: {len(df)}")
df.head()

## 3. Extract elements and flag entries with computed Seebeck / power factor data

In [ ]:
def get_elements(row):
    try:
        return set(row["atoms"]["elements"])
    except Exception:
        return set()

df["elements"] = df.apply(get_elements, axis=1)
df["n_elements"] = df["elements"].apply(len)

def has_te_data(row):
    s_n = row.get("n-Seebeck", "na")
    s_p = row.get("p-Seebeck", "na")
    return (s_n not in ("na", None)) or (s_p not in ("na", None))

df["has_te_data"] = df.apply(has_te_data, axis=1)
print(f"Entries with some Seebeck data: {df['has_te_data'].sum()}")

te_cols = [
    "jid", "formula", "spg_number", "spg_symbol", "crys",
    "optb88vdw_bandgap", "mbj_bandgap",
    "n-Seebeck", "p-Seebeck",
    "n-powerfact", "p-powerfact",
    "avg_elec_mass", "avg_hole_mass",
    "formation_energy_peratom", "ehull",
]
te_cols = [c for c in te_cols if c in df.columns]
print("Columns kept:", te_cols)

## 4. Bismuth-telluride / tetradymite family
Compounds containing a pnictogen (Bi or Sb) **and** a chalcogen (Te, Se, or S).

In [ ]:
pnictogens = {"Bi", "Sb"}
chalcogens = {"Te", "Se", "S"}

def is_bite_family(elems):
    return bool(elems & pnictogens) and bool(elems & chalcogens)

bite_mask = df["elements"].apply(is_bite_family) & df["has_te_data"]
df_bite = df.loc[bite_mask, te_cols].reset_index(drop=True)
print(f"Bi2Te3 / tetradymite-family entries with TE data: {len(df_bite)}")
df_bite.head(10)

## 5. Half-Heusler family
Ternary (3-element) compounds, 1:1:1 stoichiometry, space group F-43m (#216).

In [ ]:
def is_half_heusler(row):
    if row["n_elements"] != 3:
        return False
    try:
        spg = int(row.get("spg_number", -1))
    except (ValueError, TypeError):
        return False
    if spg != 216:  # F-43m
        return False
    try:
        comp = row["atoms"]["elements"]
        counts = {e: comp.count(e) for e in set(comp)}
        vals = list(counts.values())
        return len(vals) == 3 and max(vals) == min(vals)
    except Exception:
        return False

hh_mask = df.apply(is_half_heusler, axis=1) & df["has_te_data"]
df_hh = df.loc[hh_mask, te_cols].reset_index(drop=True)
print(f"Half-Heusler entries with TE data: {len(df_hh)}")
df_hh.head(10)

## 6. Save both families as separate CSVs

In [ ]:
df_bite.to_csv("bismuth_telluride_family_TE.csv", index=False)
df_hh.to_csv("half_heusler_family_TE.csv", index=False)

print("Saved:")
print(" - bismuth_telluride_family_TE.csv")
print(" - half_heusler_family_TE.csv")

## 7. (Optional) Download the CSVs to your computer

In [ ]:
from google.colab import files
files.download("bismuth_telluride_family_TE.csv")
files.download("half_heusler_family_TE.csv")